# Step 4: Final Validation on a Random Sample
## Human Reliability and Model Validity Analysis

This notebook evaluates the finalized annotation procedure on an independent random sample of `n = 600` paragraphs from the full dataset. Unlike the curated development sample used in Step 3, this sample was not oversampled and therefore reflects the naturally imbalanced distribution of crime-frame cases more closely.

The workflow is:

1. Load the independent random sample of 600 paragraphs.
2. Import the labels assigned independently by both human annotators.
3. Calculate raw agreement and Krippendorff's alpha before resolving disagreements.
4. Review disagreement cases and create final human consensus labels.
5. Annotate each paragraph ten times with the finalized instruction-only zero-shot prompt using Ministral at `temperature = 0.7`.
6. Determine the final LLM label through majority voting across the ten runs.
7. Flag paragraphs with less than 80% agreement across runs as LLM hardcases.
8. Compare the majority-vote LLM labels with the human consensus labels using accuracy, precision, recall, macro-F1, weighted-F1, and Cohen's kappa.

The final validity evaluation focuses on distinguishing `NO_CRIME_FRAME` from `CRIME_FRAME_NEG`. Although the finalized annotation instruction retained `CRIME_FRAME_POS`, this category was too rare in the development data to evaluate reliably.

## Imports and file paths

In [11]:
from pathlib import Path
import pandas as pd
import numpy as np
import os
import importlib
import krippendorff
from dotenv import load_dotenv
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    cohen_kappa_score,
    precision_score,
    recall_score
)

In [12]:
load_dotenv()
api_key = os.getenv("CAC_API_KEY")

HOST  = "https://maki.uni-mannheim.de/v1"
MODEL = "ministral-3-14b"

print(f"API key loaded: {'YES' if api_key else 'NO - check your .env file'}")
print(f"Host: {HOST}")
print(f"Model: {MODEL}")

API key loaded: YES
Host: https://maki.uni-mannheim.de/v1
Model: ministral-3-14b


In [13]:
#for model validity check 
import annotation_setup
importlib.reload(annotation_setup)

from annotation_setup import (
    VALID_LABELS,
    instructions,
    reminder,
    annotate,
    parse_label,
    parse_explanation
)

print("Shared annotation setup loaded.")

Shared annotation setup loaded.


In [ ]:
# ── Config ───────────────────────────────────────────────────────────────
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

# Step 4 output files
STEP4_TESTSET_FOR_HUMAN_600 = RESULTS_DIR / "step4_testset_600_for_human_annotation.csv"
STEP4_HUMAN_HARDCASES_600 = "results/step4_human_hardcases_for_discussion_600.csv"
STEP4_HUMAN_COMPLETED_600 = "results/step4_human_completed_ground_truth_600.csv"
STEP4_HUMAN_COMPLETED_WITH_MODEL_600 = "results/step4_human_completed_ground_truth_600_with_model.csv"
STEP4_HUMAN_MODEL_COMPARISON_600 = "results/step4_human_model_comparison_600.csv"
STEP4_MODEL_HARDCASES_600 = "results/step4_model_hardcases_600.csv"
STEP4_EVALUATION_600 = "results/step4_evaluation_summary_600.csv"

## Step 3.5: Human reliability and model validity checks

In this step we check the human reliability and model validity for the two 250 samples (n =500) and **the testset (n=600) for Step 4**

**Step 3.5a** creates a usable human consensus label. Because not every row was coded by all three (later two) annotators. The rule is: If at least two annotators chose the same valid label, that label becomes the human consensus. If two annotators coded a row and disagree, or if three annotators all chose different labels, the row is flagged as a human hard case and has to be discussed manually.

**Step 3.5b** checks human reliability. This tells us how consistently the human annotators applied the codebook. We report the percentage of rows with agreement among rows coded by at least two people and, where possible, pairwise Cohen's kappa between annotator pairs.

**Step 3.5c** checks model validity. This compares the model label to the human consensus label. Rows without a usable human consensus are excluded from the model evaluation until they have been resolved manually in `final_human_label`.


### Step 3.5a: Create human consensus labels and flag human hard cases

This block reads the completed human annotation file, cleans the annotator labels, checks for typos, and creates a consensus label. A consensus exists when at least two annotators agree. Rows without consensus are saved to a separate file for manual discussion. Meassuring intercoder reliability. 

In [ ]:
# ── Step 3.5a: Create human consensus labels and check reliability for testset n=600──────

human_annotation = pd.read_csv(STEP3_TESTSET_FOR_HUMAN_600)

annotator_cols = ["label_robin", "label_marmee"]

# Clean labels
for col in annotator_cols:
    human_annotation[col] = human_annotation[col].astype(str).str.strip()
    human_annotation[col] = human_annotation[col].replace({
        "": np.nan,
        "nan": np.nan,
        "None": np.nan
    })

# Check for invalid labels or typos
for col in annotator_cols:
    invalid = human_annotation[
        human_annotation[col].notna() &
        ~human_annotation[col].isin(VALID_LABELS)
    ]

    if len(invalid) > 0:
        print(f"Warning: {col} has invalid labels:")
        display(invalid[["testset_id", col]].head(20))


def get_consensus_label(row):
    robin = row["label_robin"]
    marmee = row["label_marmee"]

    if pd.isna(robin) or pd.isna(marmee):
        return np.nan

    if robin == marmee:
        return robin

    return "NO_HUMAN_CONSENSUS"


def get_human_agreement(row):
    robin = row["label_robin"]
    marmee = row["label_marmee"]

    if pd.isna(robin) or pd.isna(marmee):
        return np.nan

    return 1.0 if robin == marmee else 0.0


def get_n_annotators(row):
    return sum(pd.notna(row[col]) for col in annotator_cols)


human_annotation["n_human_annotators"] = human_annotation.apply(get_n_annotators, axis=1)
human_annotation["human_consensus_label"] = human_annotation.apply(get_consensus_label, axis=1)
human_annotation["human_agreement"] = human_annotation.apply(get_human_agreement, axis=1)

human_hard_cases = human_annotation[
    human_annotation["human_consensus_label"] == "NO_HUMAN_CONSENSUS"
].copy()

if "final_human_label" not in human_hard_cases.columns:
    human_hard_cases["final_human_label"] = ""

# ti not overwrite hardcases but add them if we annotated more cases 
if os.path.exists(STEP3_HUMAN_HARDCASES_600):
    existing_cases = pd.read_csv(STEP3_HUMAN_HARDCASES_600)
    
    # Concatenate old and new cases. Placing 'existing_cases' first combined with keep='first'
    # ensures that rows you already manually annotated in the CSV are prioritized and preserved.
    human_hard_cases = pd.concat([existing_cases, human_hard_cases]).drop_duplicates(subset=['testset_id'], keep='first')
    print(f"-> Successfully merged new hard cases with existing annotations from '{STEP3_HUMAN_HARDCASES_600}'.")
else:
    print(f"-> No existing file found. Creating a brand new tracking template at '{STEP3_HUMAN_HARDCASES_600}'.")

human_hard_cases.to_csv(STEP3_HUMAN_HARDCASES_600, index=False)

print("=== Human consensus creation ===")

print("Number of annotators per row:")
print(human_annotation["n_human_annotators"].value_counts(dropna=False).sort_index())

print("\nConsensus label distribution:")
print(human_annotation["human_consensus_label"].value_counts(dropna=False))

double_coded = human_annotation[human_annotation["n_human_annotators"] == 2].copy()

n_total = len(double_coded)
n_agree = double_coded["human_consensus_label"].isin(VALID_LABELS).sum()
n_disagree = (double_coded["human_consensus_label"] == "NO_HUMAN_CONSENSUS").sum()

percent_agreement = n_agree / n_total if n_total > 0 else np.nan
percent_disagreement = n_disagree / n_total if n_total > 0 else np.nan

print("\n=== Human reliability summary ===")
print(f"Double-coded rows: {n_total}")
print(f"Rows with agreement: {n_agree} ({percent_agreement * 100:.1f}%)")
print(f"Rows without agreement: {n_disagree} ({percent_disagreement * 100:.1f}%)")

print("\nHuman agreement distribution among double-coded rows:")
print(double_coded["human_agreement"].describe().round(2))

# Cohen's kappa and Krippendorff's alpha between the two annotators
pair_data = human_annotation[
    human_annotation["label_robin"].isin(VALID_LABELS) &
    human_annotation["label_marmee"].isin(VALID_LABELS)
].copy()

if len(pair_data) > 0:
    pair_accuracy = accuracy_score(pair_data["label_robin"], pair_data["label_marmee"])
    pair_kappa = cohen_kappa_score(
        pair_data["label_robin"],
        pair_data["label_marmee"],
        labels=VALID_LABELS
    )
    
    # Prepare data structure for Krippendorff (a list of lists/matrix where rows=coders, columns=items)
    reliability_data = [
        pair_data["label_robin"].tolist(),
        pair_data["label_marmee"].tolist()
    ]
    pair_alpha = krippendorff.alpha(reliability_data=reliability_data, level_of_measurement="nominal")
else:
    pair_accuracy = np.nan
    pair_kappa = np.nan
    pair_alpha = np.nan

print("\n=== Pairwise human reliability ===")
print(f"Overlapping valid annotations: {len(pair_data)}")
print(f"Percent agreement: {pair_accuracy * 100:.1f}%")
print(f"Cohen's kappa: {pair_kappa:.3f}") # can be excluded bc use of Krippendorffs 
print(f"Krippendorff's alpha (nominal): {pair_alpha:.3f}")

print(f"\nHuman hard cases without consensus: {len(human_hard_cases)}")
print(f"Saved human hard cases to {STEP3_HUMAN_HARDCASES_600}")

-> Successfully merged new hard cases with existing annotations from 'results/human_hardcases_for_discussion_600.csv'.
=== Human consensus creation ===
Number of annotators per row:
n_human_annotators
2    600
Name: count, dtype: int64

Consensus label distribution:
human_consensus_label
NO_CRIME_FRAME        546
NO_HUMAN_CONSENSUS     34
CRIME_FRAME_NEG        20
Name: count, dtype: int64

=== Human reliability summary ===
Double-coded rows: 600
Rows with agreement: 566 (94.3%)
Rows without agreement: 34 (5.7%)

Human agreement distribution among double-coded rows:
count    600.00
mean       0.94
std        0.23
min        0.00
25%        1.00
50%        1.00
75%        1.00
max        1.00
Name: human_agreement, dtype: float64

=== Pairwise human reliability ===
Overlapping valid annotations: 600
Percent agreement: 94.3%
Cohen's kappa: 0.516
Krippendorff's alpha (nominal): 0.511

Human hard cases without consensus: 38
Saved human hard cases to results/human_hardcases_for_discussion_6

In [ ]:
# ── After solving hard cases: Create final human completed ground truth for n=600 ──

human_annotation = pd.read_csv(STEP3_TESTSET_FOR_HUMAN_600)
human_hard_cases = pd.read_csv(STEP3_HUMAN_HARDCASES_600)

human_annotation.columns = human_annotation.columns.str.strip()
human_hard_cases.columns = human_hard_cases.columns.str.strip()

annotator_cols = ["label_robin", "label_marmee"]


def clean_label_series(series):
    return (
        series
        .astype(str)
        .str.strip()
        .replace({
            "": np.nan,
            "nan": np.nan,
            "NaN": np.nan,
            "None": np.nan,
            "NONE": np.nan
        })
    )


def normalize_testset_id(series):
    numeric = pd.to_numeric(series, errors="coerce")
    return numeric.astype("Int64").astype(str).replace("<NA>", np.nan)


# Drop completely empty rows from the annotation file
human_annotation = human_annotation.dropna(how="all").copy()

# Clean testset_id
human_annotation["testset_id"] = normalize_testset_id(human_annotation["testset_id"])
human_hard_cases["testset_id"] = normalize_testset_id(human_hard_cases["testset_id"])

# Drop rows without a real testset_id
human_annotation = human_annotation[human_annotation["testset_id"].notna()].copy()
human_hard_cases = human_hard_cases[human_hard_cases["testset_id"].notna()].copy()

# Clean annotator labels
for df_tmp in [human_annotation, human_hard_cases]:
    for col in annotator_cols:
        if col in df_tmp.columns:
            df_tmp[col] = clean_label_series(df_tmp[col])

# Clean final hard case labels
if "final_human_label" not in human_hard_cases.columns:
    raise ValueError(
        f"'final_human_label' column is missing in {STEP3_HUMAN_HARDCASES_600}"
    )

human_hard_cases["final_human_label"] = clean_label_series(
    human_hard_cases["final_human_label"]
)

# Recreate consensus labels
human_annotation["human_consensus_label"] = human_annotation.apply(
    get_consensus_label,
    axis=1
)

# Keep only solved hard cases
resolved_hard_cases = human_hard_cases[
    human_hard_cases["final_human_label"].isin(VALID_LABELS)
][["testset_id", "final_human_label"]].copy()

resolved_hard_cases = resolved_hard_cases.drop_duplicates(
    subset=["testset_id"],
    keep="last"
)

# Map manually resolved hard case labels into full file
final_label_map = resolved_hard_cases.set_index("testset_id")["final_human_label"]

human_completed = human_annotation.copy()

human_completed["final_human_label_from_hardcases"] = (
    human_completed["testset_id"].map(final_label_map)
)

# Use hardcase final_human_label first, otherwise use consensus
human_completed["final_human_label"] = human_completed[
    "final_human_label_from_hardcases"
].fillna(
    human_completed["human_consensus_label"]
)

# Keep only rows with valid final labels
human_completed = human_completed[
    human_completed["final_human_label"].isin(VALID_LABELS)
].copy()

human_completed = human_completed.drop(
    columns=["final_human_label_from_hardcases"]
)

human_completed.to_csv(STEP3_HUMAN_COMPLETED_600, index=False)

print(f"Saved completed human ground truth to {STEP3_HUMAN_COMPLETED_600}")
print(f"Rows in original 600 file after dropping empty rows: {len(human_annotation)}")
print(f"Rows with solved hard cases: {len(resolved_hard_cases)}")
print(f"Rows in completed ground truth: {len(human_completed)}")

print("\nFinal human label distribution:")
print(human_completed["final_human_label"].value_counts())

missing_final = len(human_annotation) - len(human_completed)

print(f"\nRows without final valid human label: {missing_final}")

if missing_final > 0:
    unresolved = human_annotation.merge(
        human_completed[["testset_id"]],
        on="testset_id",
        how="left",
        indicator=True
    )

    unresolved = unresolved[
        unresolved["_merge"] == "left_only"
    ].drop(columns=["_merge"])

    print("\nUnresolved rows preview:")
    display(
        unresolved[
            ["testset_id", "label_robin", "label_marmee", "human_consensus_label"]
        ].head(30)
    )

Saved completed human ground truth to results/human_completed_ground_truth_600.csv
Rows in original new 250 file after dropping empty rows: 600
Rows with solved hard cases: 34
Rows in completed ground truth: 596

Final human label distribution:
final_human_label
NO_CRIME_FRAME     564
CRIME_FRAME_NEG     32
Name: count, dtype: int64

Rows without final valid human label: 4

Unresolved rows preview:


,testset_id,label_robin,label_marmee,human_consensus_label
3,4,NO_CRIME_FRAME,CRIME_FRAME_NEG,NO_HUMAN_CONSENSUS
7,8,NO_CRIME_FRAME,CRIME_FRAME_NEG,NO_HUMAN_CONSENSUS
24,25,NO_CRIME_FRAME,CRIME_FRAME_NEG,NO_HUMAN_CONSENSUS
277,278,CRIME_FRAME_NEG,NO_CRIME_FRAME,NO_HUMAN_CONSENSUS


## 3.5b Rerun Model Labels on Human Completed Cases

This section reruns the model on exactly the same cases that were completed by the human annotators.

The model only receives the paragraph text, the current annotation instruction from `annotation_setup.py`, and the reminder. It does not receive the human labels. This keeps the model annotation blind to the ground truth.

Each time this cell runs, the existing `model_label` column gets overwritten for the human completed cases. The cell also updates the model explanation, raw model output, and model name. This makes sure that the internal model info file always reflects the newest annotation instruction and the currently selected model.

In [22]:
# Step 3.5b: Rerun model and overwrite model columns for completed n=600

human_completed = pd.read_csv(STEP3_HUMAN_COMPLETED_600)
human_completed.columns = human_completed.columns.str.strip()

def normalize_testset_id(series):
    numeric = pd.to_numeric(series, errors="coerce")
    return numeric.astype("Int64").astype(str).replace("<NA>", np.nan)

human_completed["testset_id"] = normalize_testset_id(human_completed["testset_id"])
human_completed = human_completed[human_completed["testset_id"].notna()].copy()

human_completed = human_completed[
    human_completed["final_human_label"].isin(VALID_LABELS)
].copy().reset_index(drop=True)

print(f"Human completed rows with valid final label: {len(human_completed)}")
print(f"Running fresh model labels with model: {MODEL}")

required_cols = [
    "testset_id",
    "text_block_en",
    "final_human_label"
]

missing_cols = [
    col for col in required_cols
    if col not in human_completed.columns
]

if missing_cols:
    raise ValueError(
        f"Missing columns in {STEP3_HUMAN_COMPLETED_600}: {missing_cols}"
    )

for col in ["model_label", "model_explanation", "model_raw_output", "model"]:
    if col not in human_completed.columns:
        human_completed[col] = pd.NA

fresh_labels = {}

for i, row in human_completed.iterrows():
    raw = annotate(
        text=str(row["text_block_en"]),
        instructions=instructions,
        reminder=reminder,
        api_key=api_key,
        HOST=HOST,
        MODEL=MODEL,
        temperature=0.0
    )

    fresh_labels[row["testset_id"]] = {
        "model_label": parse_label(raw),
        "model_explanation": parse_explanation(raw),
        "model_raw_output": raw,
        "model": MODEL
    }

    if (i + 1) % 10 == 0 or (i + 1) == len(human_completed):
        print(f"[{i + 1}/{len(human_completed)}] done")

for testset_id, values in fresh_labels.items():
    mask = human_completed["testset_id"] == testset_id

    human_completed.loc[mask, "model_label"] = values["model_label"]
    human_completed.loc[mask, "model_explanation"] = values["model_explanation"]
    human_completed.loc[mask, "model_raw_output"] = values["model_raw_output"]
    human_completed.loc[mask, "model"] = values["model"]

human_completed.to_csv(STEP3_HUMAN_COMPLETED_WITH_MODEL_600, index=False)

print(f"\nSaved completed new 250 file with fresh model labels to {STEP3_HUMAN_COMPLETED_WITH_MODEL_600}")

print("\nUpdated model label distribution:")
print(
    human_completed["model_label"].value_counts(dropna=False)
)

print("\nFinal human label distribution:")
print(
    human_completed["final_human_label"].value_counts(dropna=False)
)

Human completed rows with valid final label: 596
Running fresh model labels with model: ministral-3-14b
[10/596] done
[20/596] done
[30/596] done
[40/596] done
[50/596] done
[60/596] done
[70/596] done
[80/596] done
[90/596] done
[100/596] done
[110/596] done
[120/596] done
[130/596] done
[140/596] done
[150/596] done
[160/596] done
[170/596] done
[180/596] done
[190/596] done
[200/596] done
[210/596] done
[220/596] done
[230/596] done
[240/596] done
[250/596] done
[260/596] done
[270/596] done
[280/596] done
[290/596] done
[300/596] done
[310/596] done
[320/596] done
[330/596] done
[340/596] done
[350/596] done
[360/596] done
[370/596] done
[380/596] done
[390/596] done
[400/596] done
[410/596] done
[420/596] done
[430/596] done
[440/596] done
[450/596] done
[460/596] done
[470/596] done
[480/596] done
[490/596] done
[500/596] done
[510/596] done
[520/596] done
[530/596] done
[540/596] done
[550/596] done
[560/596] done
[570/596] done
[580/596] done
[590/596] done
[596/596] done

Save

## 3.5c Validate Updated Model Labels Against Human Ground Truth

This section compares the updated model labels against the final human labels.

The human label `final_human_label` is treated as the ground truth. The model label `model_label` comes from the updated internal model info file created in Step 3.5b.

The section creates two output files. The first file contains all human completed cases with both the human label and the model label, including whether they agree or disagree. The second file contains only the disagreement cases and is used as the model hardcase file for error analysis.

The section also calculates accuracy, macro precision, macro recall, macro F1, weighted F1, and Cohen’s kappa, to check model validity. The metric summary is saved to `step3_evaluation_summary.csv`.

In [ ]:
# ── Step 3.5c: Validity checks against human ground truth for n=600 excluded crime_frame_pos ─────


# Labels used for this binary evaluation
EVAL_LABELS = [
    "NO_CRIME_FRAME",
    "CRIME_FRAME_NEG"
]


# Load completed human annotations with fresh model labels
comparison = pd.read_csv(STEP3_HUMAN_COMPLETED_WITH_MODEL_600)

# Remove accidental whitespace from column names
comparison.columns = comparison.columns.str.strip()


def clean_label_series(series):
    """
    Clean annotation labels and convert empty values to missing values.
    """
    return (
        series
        .astype("string")
        .str.strip()
        .replace({
            "": pd.NA,
            "nan": pd.NA,
            "NaN": pd.NA,
            "None": pd.NA,
            "NONE": pd.NA
        })
    )


def normalize_testset_id(series):
    """
    Convert testset_id values to normalized integer-like strings.
    """
    numeric = pd.to_numeric(series, errors="coerce")

    return (
        numeric
        .astype("Int64")
        .astype("string")
        .replace("<NA>", pd.NA)
    )


# Required columns
required_cols = [
    "testset_id",
    "final_human_label",
    "model_label"
]

missing_cols = [
    col
    for col in required_cols
    if col not in comparison.columns
]

if missing_cols:
    raise ValueError(
        f"Missing columns in "
        f"{STEP3_HUMAN_COMPLETED_WITH_MODEL_600}: "
        f"{missing_cols}"
    )


# Clean identifiers and labels
comparison["testset_id"] = normalize_testset_id(
    comparison["testset_id"]
)

comparison["final_human_label"] = clean_label_series(
    comparison["final_human_label"]
)

comparison["model_label"] = clean_label_series(
    comparison["model_label"]
)


# Remove rows without a valid testset_id
comparison = (
    comparison[
        comparison["testset_id"].notna()
    ]
    .copy()
    .reset_index(drop=True)
)


# Keep rows with a valid final human label
comparison = (
    comparison[
        comparison["final_human_label"].isin(EVAL_LABELS)
    ]
    .copy()
    .reset_index(drop=True)
)


# Mark whether the model label is valid for this evaluation
comparison["model_valid_label"] = (
    comparison["model_label"].isin(EVAL_LABELS)
)


# Human and model agreement
comparison["human_model_agree"] = (
    comparison["final_human_label"]
    == comparison["model_label"]
)

comparison["agreement_status"] = np.where(
    comparison["human_model_agree"],
    "AGREE",
    "DISAGREE"
)


# Arrange important columns first
preferred_columns = [
    "testset_id",
    "group",
    "text_block_en",
    "label_robin",
    "label_marmee",
    "human_consensus_label",
    "final_human_label",
    "model",
    "model_label",
    "model_explanation",
    "model_raw_output",
    "model_valid_label",
    "human_model_agree",
    "agreement_status",
    "article_id",
    "par_index"
]

available_preferred_columns = [
    col
    for col in preferred_columns
    if col in comparison.columns
]

remaining_columns = [
    col
    for col in comparison.columns
    if col not in available_preferred_columns
]

comparison = comparison[
    available_preferred_columns + remaining_columns
].copy()


# Save complete comparison file
comparison.to_csv(
    STEP3_HUMAN_MODEL_COMPARISON_600,
    index=False
)

print(
    f"Saved human model comparison to "
    f"{STEP3_HUMAN_MODEL_COMPARISON_600}"
)


# Agreement counts
print("\nAgreement counts:")

print(
    comparison["agreement_status"]
    .value_counts(dropna=False)
)


# Save model disagreements
model_hardcases = comparison[
    comparison["agreement_status"] == "DISAGREE"
].copy()

model_hardcases.to_csv(
    STEP3_MODEL_HARDCASES_600,
    index=False
)

print(
    f"\nSaved model hardcases to "
    f"{STEP3_MODEL_HARDCASES_600}"
)

print(
    f"Number of model hardcases: "
    f"{len(model_hardcases)}"
)


# Keep only rows where both labels are valid
eval_data = comparison[
    comparison["final_human_label"].isin(EVAL_LABELS)
    & comparison["model_label"].isin(EVAL_LABELS)
].copy()


print("\nModel validity against human ground truth")

print(
    f"Rows used for model evaluation: "
    f"{len(eval_data)}"
)

print(
    f"Rows excluded: "
    f"{len(comparison) - len(eval_data)}"
)


# Label distributions
print("\nHuman label distribution:")

print(
    eval_data["final_human_label"]
    .value_counts(dropna=False)
)


print("\nModel label distribution:")

print(
    eval_data["model_label"]
    .value_counts(dropna=False)
)


if len(eval_data) == 0:
    raise ValueError(
        "No valid rows available for evaluation. "
        "Check final_human_label and model_label values."
    )


# Ground truth and predictions
y_true = eval_data["final_human_label"]
y_pred = eval_data["model_label"]


# Overall accuracy
accuracy = accuracy_score(
    y_true,
    y_pred
)


# Macro metrics across the two observed classes
macro_precision = precision_score(
    y_true,
    y_pred,
    labels=EVAL_LABELS,
    average="macro",
    zero_division=0
)

macro_recall = recall_score(
    y_true,
    y_pred,
    labels=EVAL_LABELS,
    average="macro",
    zero_division=0
)

macro_f1 = f1_score(
    y_true,
    y_pred,
    labels=EVAL_LABELS,
    average="macro",
    zero_division=0
)


# Weighted F1
weighted_f1 = f1_score(
    y_true,
    y_pred,
    labels=EVAL_LABELS,
    average="weighted",
    zero_division=0
)


# Cohen's kappa
kappa = cohen_kappa_score(
    y_true,
    y_pred,
    labels=EVAL_LABELS
)


# Positive class metrics
positive_precision = precision_score(
    y_true,
    y_pred,
    labels=EVAL_LABELS,
    pos_label="CRIME_FRAME_NEG",
    average="binary",
    zero_division=0
)

positive_recall = recall_score(
    y_true,
    y_pred,
    labels=EVAL_LABELS,
    pos_label="CRIME_FRAME_NEG",
    average="binary",
    zero_division=0
)

positive_f1 = f1_score(
    y_true,
    y_pred,
    labels=EVAL_LABELS,
    pos_label="CRIME_FRAME_NEG",
    average="binary",
    zero_division=0
)


# Print summary metrics
print("\nSummary metrics:")

print(f"Accuracy: {accuracy:.3f}")
print(f"Macro precision: {macro_precision:.3f}")
print(f"Macro recall: {macro_recall:.3f}")
print(f"Macro F1: {macro_f1:.3f}")
print(f"Weighted F1: {weighted_f1:.3f}")
print(f"Cohen's kappa: {kappa:.3f}")

print("\nCRIME_FRAME_NEG metrics:")

print(f"Precision: {positive_precision:.3f}")
print(f"Recall: {positive_recall:.3f}")
print(f"F1: {positive_f1:.3f}")


# Confusion matrix
print("\nConfusion matrix:")

conf_matrix = confusion_matrix(
    y_true,
    y_pred,
    labels=EVAL_LABELS
)

conf_matrix_df = pd.DataFrame(
    conf_matrix,
    index=[
        f"human_{label}"
        for label in EVAL_LABELS
    ],
    columns=[
        f"model_{label}"
        for label in EVAL_LABELS
    ]
)

display(conf_matrix_df)


# Classification report
print("\nClassification report:")

print(
    classification_report(
        y_true,
        y_pred,
        labels=EVAL_LABELS,
        target_names=EVAL_LABELS,
        zero_division=0
    )
)


# Determine model name for the evaluation summary
if "MODEL" in globals():
    evaluation_model = MODEL

elif (
    "model" in comparison.columns
    and comparison["model"].notna().any()
):
    evaluation_model = (
        comparison["model"]
        .dropna()
        .iloc[0]
    )

else:
    evaluation_model = None


# Create evaluation summary
evaluation_summary = pd.DataFrame([{
    "model": evaluation_model,
    "evaluation_labels": "|".join(EVAL_LABELS),
    "n_total_human_completed": len(comparison),
    "n_eval": len(eval_data),
    "n_excluded": len(comparison) - len(eval_data),
    "n_agree": int(
        comparison["human_model_agree"].sum()
    ),
    "n_disagree": int(
        (~comparison["human_model_agree"]).sum()
    ),
    "agreement_rate": (
        comparison["human_model_agree"].mean()
    ),
    "accuracy": accuracy,
    "macro_precision": macro_precision,
    "macro_recall": macro_recall,
    "macro_f1": macro_f1,
    "weighted_f1": weighted_f1,
    "crime_frame_neg_precision": positive_precision,
    "crime_frame_neg_recall": positive_recall,
    "crime_frame_neg_f1": positive_f1,
    "cohens_kappa": kappa
}])


# Save evaluation summary
evaluation_summary.to_csv(
    STEP3_EVALUATION_600,
    index=False
)

print(
    f"\nSaved evaluation summary to "
    f"{STEP3_EVALUATION_600}"
)

Saved human model comparison to results/human_model_comparison_600.csv

Agreement counts:
agreement_status
AGREE       575
DISAGREE     21
Name: count, dtype: int64

Saved model hardcases to results/model_hardcases_600.csv
Number of model hardcases: 21

Model validity against human ground truth
Rows used for model evaluation: 596
Rows excluded: 0

Human label distribution:
final_human_label
NO_CRIME_FRAME     564
CRIME_FRAME_NEG     32
Name: count, dtype: Int64

Model label distribution:
model_label
NO_CRIME_FRAME     563
CRIME_FRAME_NEG     33
Name: count, dtype: Int64

Summary metrics:
Accuracy: 0.965
Macro precision: 0.824
Macro recall: 0.834
Macro F1: 0.829
Weighted F1: 0.965
Cohen's kappa: 0.658

CRIME_FRAME_NEG metrics:
Precision: 0.667
Recall: 0.688
F1: 0.677

Confusion matrix:


,model_NO_CRIME_FRAME,model_CRIME_FRAME_NEG
human_NO_CRIME_FRAME,553,11
human_CRIME_FRAME_NEG,10,22



Classification report:
                 precision    recall  f1-score   support

 NO_CRIME_FRAME       0.98      0.98      0.98       564
CRIME_FRAME_NEG       0.67      0.69      0.68        32

       accuracy                           0.96       596
      macro avg       0.82      0.83      0.83       596
   weighted avg       0.97      0.96      0.97       596


Saved evaluation summary to results/evaluation_summary_600.csv
